# 01 — Data Understanding & Quality Assessment

**Owner:** Mkhawana Thamelia Hleketani (Data Engineer / Big Data)

Loads the verified Kaggle dataset and profiles it: shape, dtypes, missing values, duplicates, invalid ranges, target balance and feature redundancy. Nothing is deleted here — this step only reports.

In [1]:
# Make the src package importable whether or not it is pip-installed.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(ROOT)
print('project root:', ROOT)

project root: C:\Users\johnm\Big_Data_Challenge_Project


In [2]:
from churn_project import data_loading, data_validation as dv
df = data_loading.load_raw_pandas()
print('shape:', df.shape)
df.head()

shape: (10000, 15)


,customer_id,signup_date,last_purchase_date,days_since_last_purchase,account_age_days,country,age,gender,device_type,favorite_category,total_orders,total_spent_usd,avg_order_value,is_premium_member,churned
0,CUST007563,2024-01-01,2024-02-02,699,32,United States,39,Female,Mobile,Groceries,4,286.08,71.52,0,1
1,CUST007904,2024-01-01,2024-06-28,552,179,France,26,Other,Mobile,Beauty,79,268.42,3.40,0,1
2,CUST005044,2024-01-01,2024-04-29,612,119,Japan,46,Female,Mobile,Fashion,1,60.28,60.28,0,1
3,CUST006602,2024-01-01,2025-03-25,282,449,Germany,34,Female,Tablet,Groceries,1,139.94,139.94,1,1
4,CUST004196,2024-01-01,2024-06-28,552,179,United States,36,Female,Mobile,Home & Garden,1,10.00,10.00,1,1


## Schema check and dtypes

In [3]:
print(dv.validate_schema(df))
df.dtypes

{'missing_columns': [], 'unexpected_columns': []}


customer_id                         object
signup_date                 datetime64[ns]
last_purchase_date          datetime64[ns]
days_since_last_purchase             int64
account_age_days                     int64
country                             object
age                                  int64
gender                              object
device_type                         object
favorite_category                   object
total_orders                         int64
total_spent_usd                    float64
avg_order_value                    float64
is_premium_member                    int64
churned                              int64
dtype: object

## Missing values and duplicates

In [4]:
display(dv.missing_value_report(df))
print(dv.duplicate_report(df))

,column,missing_count,missing_pct
0,customer_id,0,0.0
1,signup_date,0,0.0
2,last_purchase_date,0,0.0
3,days_since_last_purchase,0,0.0
4,account_age_days,0,0.0
5,country,0,0.0
6,age,0,0.0
7,gender,0,0.0
8,device_type,0,0.0
9,favorite_category,0,0.0


{'duplicate_rows': 0, 'duplicate_customer_ids': 0}


## Invalid values and numeric summary

In [5]:
display(dv.invalid_value_report(df))
display(dv.numeric_summary(df))

,column,rule,n_violations
0,age,age < 0 or age > 120,0
1,days_since_last_purchase,negative,0
2,account_age_days,negative,0
3,total_orders,negative,0
4,total_spent_usd,negative,0
5,avg_order_value,negative,0
6,is_premium_member,"not in {0,1}",0
7,churned,"not in {0,1}",0


,column,count,mean,std,min,25%,50%,75%,max
0,days_since_last_purchase,10000.0,118.835500,169.226426,1.00,1.0000,1.000,208.0000,719.00
1,account_age_days,10000.0,240.763900,171.295808,0.00,97.0000,210.000,362.0000,725.00
2,age,10000.0,33.885600,9.465531,18.00,27.0000,34.000,40.0000,78.00
3,total_orders,10000.0,7.713700,29.907347,1.00,1.0000,1.000,3.0000,300.00
4,total_spent_usd,10000.0,312.222272,530.249651,10.00,65.7650,151.000,343.1775,9564.94
5,avg_order_value,10000.0,208.596229,443.111121,0.03,26.0875,78.335,211.3050,9564.94
6,is_premium_member,10000.0,0.143900,0.351006,0.00,0.0000,0.000,0.0000,1.00


## Target distribution and redundancy

In [6]:
display(dv.target_distribution(df))
display(dv.redundancy_report(df))

,class,count,proportion
0,0,6229,0.6229
1,1,3771,0.3771


""


**Findings:** 10,000 x 15, one row per customer, no missing values, no duplicates. Target 37.7% churn. `total_spent_usd` and `avg_order_value` are redundant (r=0.85).